# NIST23 Data Quality Analysis

Explores peak-position heuristics and their relationship to DAG coverage.
The analysis uses mbfs_prepass wrecall and is therefore model-dependent; it can identify
candidates for follow-up, but it cannot by itself prove that a NIST spectrum is bad.

In [ ]:
import io
import tarfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src/fragnnet").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from within the frag-gnn repository")


REPO_ROOT = find_repo_root()
FRAG_DP = REPO_ROOT / "data/frag"
PROC_DP = REPO_ROOT / "data/proc/nist23"

In [ ]:
# Load mbfs_prepass wrecall columns
MBFS_PREPASS_PATH = FRAG_DP / "nist23_ma_mi_d3_h4_isoFalse_mbfs_prepass"

W4 = "wrecall_0.01_h4"
W2 = "wrecall_0.01_h2"

spec_stats_path = MBFS_PREPASS_PATH / "spec_stats_df.pkl"
if not spec_stats_path.exists():
    raise FileNotFoundError(
        f"Missing {spec_stats_path}. Generate the NIST23 mbfs_prepass statistics before "
        "running this notebook."
    )
pp_stats = pd.read_pickle(spec_stats_path)[["spec_id", W4, W2]].copy()
pp_stats["spec_id"] = pp_stats["spec_id"].astype(int)
print(f"pp_stats: {pp_stats.shape}")

In [ ]:
# Load nist23 spec metadata
spec_meta = pd.read_pickle(PROC_DP / "spec_df.pkl")[
    ["spec_id", "mol_id", "inst_type", "frag_mode", "prec_type",
     "ion_mode", "ace", "nce", "peaks"]
].copy()
spec_meta["num_peaks"] = spec_meta["peaks"].apply(
    lambda x: len(x) if x is not None else 0
)
spec_meta = spec_meta.drop(columns=["peaks"])
print(f"spec_meta: {spec_meta.shape}")

# Join and create DAG-coverage tiers by wrecall_h4
merged = pp_stats.merge(spec_meta, on="spec_id", how="left")
print(f"merged: {merged.shape}  unmatched: {merged['inst_type'].isna().sum():,}")

COVERAGE_TIER_ORDER = ["very_low (<0.1)", "low (0.1-0.3)", "medium (0.3-0.5)", "good (>0.5)"]
merged["coverage_tier"] = pd.cut(
    merged[W4],
    bins=[-0.001, 0.1, 0.3, 0.5, 1.001],
    labels=COVERAGE_TIER_ORDER,
)
COVERAGE_TIER_COLORS = {
    "very_low (<0.1)": "tab:red",
    "low (0.1-0.3)": "tab:orange",
    "medium (0.3-0.5)": "tab:blue",
    "good (>0.5)": "tab:green",
}
print("\nDAG-coverage tier counts:")
print(merged["coverage_tier"].value_counts().sort_index())

## NIST Data Quality Analysis

Exploratory diagnostics relating peak positions to DAG coverage. These heuristics do not establish intrinsic spectrum quality or prove that a spectrum is bad.

In [ ]:
# A. Mol metadata join + B/D. Peak quality feature extraction
# ---------------------------------------------------------------
# Joins exact_mw / num_atoms from mol_df, then computes per-spectrum:
#   frac_not_far_above_prec – fraction of peaks with m/z ≤ prec_mz + 1.5
#   n_far_above_prec        – count of peaks with m/z > prec_mz + 2 Da
# These are mass-position heuristics, not molecular-formula checks or definitive noise labels.
#   has_prec_peak     – any peak within 0.02 Da of prec_mz (unfragmented leakthrough)
#   intensity_entropy – -Σ p·log(p) over normalised intensities

mol_df_meta = pd.read_pickle(PROC_DP / "mol_df.pkl")[["mol_id", "formula", "exact_mw", "num_atoms"]]
merged = merged.merge(mol_df_meta, on="mol_id", how="left")

spec_ids_needed = set(merged["spec_id"].values)
spec_sub = pd.read_pickle(PROC_DP / "spec_df.pkl")[["spec_id", "prec_mz", "peaks"]]
spec_sub = spec_sub[spec_sub["spec_id"].isin(spec_ids_needed)].reset_index(drop=True)


def compute_peak_feats(row: pd.Series) -> tuple:
    peaks = row["peaks"]
    prec_mz = row["prec_mz"]
    if not peaks or prec_mz != prec_mz:  # empty or NaN
        return (np.nan, np.nan, False, np.nan)
    mzs = np.fromiter((p[0] for p in peaks), dtype=float, count=len(peaks))
    ints = np.fromiter((p[1] for p in peaks), dtype=float, count=len(peaks))
    frac_not_far_above = float((mzs <= prec_mz + 1.5).mean())
    n_far_above = int((mzs > prec_mz + 2.0).sum())
    has_prec = bool((np.abs(mzs - prec_mz) <= 0.02).any())
    intensity_sum = ints.sum()
    if not np.isfinite(intensity_sum) or intensity_sum <= 0:
        return (frac_not_far_above, n_far_above, has_prec, np.nan)
    p = ints / intensity_sum
    p = p[p > 0]
    entropy = float(-np.sum(p * np.log(p)))
    return (frac_not_far_above, n_far_above, has_prec, entropy)


print("Computing peak features…")
feat_cols = ["frac_not_far_above_prec", "n_far_above_prec", "has_prec_peak", "intensity_entropy"]
feat_df = spec_sub.apply(compute_peak_feats, axis=1, result_type="expand")
feat_df.columns = feat_cols
feat_df["spec_id"] = spec_sub["spec_id"].values
merged = merged.merge(feat_df[["spec_id"] + feat_cols], on="spec_id", how="left")
print("Done. merged shape:", merged.shape)
print(merged[feat_cols].describe().round(3))

In [ ]:
# D. Peak-position diagnostics by DAG-coverage tier
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# (1) relative-to-precursor mass-position heuristic
ax = axes[0]
data = [merged.loc[merged["coverage_tier"] == t, "frac_not_far_above_prec"].dropna().values for t in COVERAGE_TIER_ORDER]
parts = ax.violinplot(data, positions=range(len(COVERAGE_TIER_ORDER)), showmedians=True, showextrema=False)
for i, (pc, t) in enumerate(zip(parts["bodies"], COVERAGE_TIER_ORDER)):
    pc.set_facecolor(COVERAGE_TIER_COLORS[t])
    pc.set_alpha(0.7)
ax.set_xticks(range(len(COVERAGE_TIER_ORDER)))
ax.set_xticklabels(COVERAGE_TIER_ORDER, rotation=15, ha="right", fontsize=8)
ax.set_ylabel("Fraction of peaks ≤ prec_mz + 1.5")
ax.set_title("Relative-to-precursor peak-position heuristic")

# (2) intensity_entropy violin
ax = axes[1]
data = [merged.loc[merged["coverage_tier"] == t, "intensity_entropy"].dropna().values for t in COVERAGE_TIER_ORDER]
parts = ax.violinplot(data, positions=range(len(COVERAGE_TIER_ORDER)), showmedians=True, showextrema=False)
for pc, t in zip(parts["bodies"], COVERAGE_TIER_ORDER):
    pc.set_facecolor(COVERAGE_TIER_COLORS[t])
    pc.set_alpha(0.7)
ax.set_xticks(range(len(COVERAGE_TIER_ORDER)))
ax.set_xticklabels(COVERAGE_TIER_ORDER, rotation=15, ha="right", fontsize=8)
ax.set_ylabel("Intensity entropy (−Σ p·log p)")
ax.set_title("Spectrum intensity entropy")

# (3) has_prec_peak & n_above_prec rates as grouped bar
ax = axes[2]
rates = {}
rates["has_prec_peak (%)"] = [
    merged.loc[merged["coverage_tier"] == t, "has_prec_peak"].mean() * 100 for t in COVERAGE_TIER_ORDER
]
rates["n_far_above_prec > 0 (%)"] = [
    (merged.loc[merged["coverage_tier"] == t, "n_far_above_prec"] > 0).mean() * 100 for t in COVERAGE_TIER_ORDER
]
x = np.arange(len(COVERAGE_TIER_ORDER))
width = 0.35
ax.bar(x - width / 2, rates["has_prec_peak (%)"], width, label="has prec peak", color="steelblue")
ax.bar(x + width / 2, rates["n_far_above_prec > 0 (%)"], width, label="peak > precursor + 2 Da", color="tomato")
ax.set_xticks(x)
ax.set_xticklabels(COVERAGE_TIER_ORDER, rotation=15, ha="right", fontsize=8)
ax.set_ylabel("% of spectra")
ax.set_title("Precursor peak and high-m/z diagnostics")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# C. Quadrant: DAG coverage vs relative-to-precursor peak-position heuristic
# x-axis: frac_not_far_above_prec (descriptive heuristic; not an intrinsic quality label)
# y-axis: wrecall_h2        (DAG coverage — model quality)
# Quadrant thresholds: x > 0.95 = "clean spectrum", y > 0.5 = "good DAG coverage"
THRESHOLD_X = 0.95
THRESHOLD_Y = 0.5

plot_df = merged[["frac_not_far_above_prec", W2, "coverage_tier"]].dropna()
# Sample for scatter legibility
sample = plot_df.sample(min(30_000, len(plot_df)), random_state=42)

fig, ax = plt.subplots(figsize=(7, 6))
for tier in COVERAGE_TIER_ORDER:
    sub = sample[sample["coverage_tier"] == tier]
    ax.scatter(sub["frac_not_far_above_prec"], sub[W2], s=2, alpha=0.3, color=COVERAGE_TIER_COLORS[tier], label=tier)

ax.axvline(THRESHOLD_X, color="k", lw=1, ls="--")
ax.axhline(THRESHOLD_Y, color="k", lw=1, ls="--")

# Quadrant labels
ax.text(0.02, 0.97, "high DAG coverage\nlow mass-position heuristic", transform=ax.transAxes,
        ha="left", va="top", fontsize=9, color="navy",
        bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", alpha=0.8))
ax.text(0.97, 0.97, "\u2705 good", transform=ax.transAxes,
        ha="right", va="top", fontsize=9, color="green",
        bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", alpha=0.8))
ax.text(0.02, 0.03, "low DAG coverage\nlow mass-position heuristic", transform=ax.transAxes,
        ha="left", va="bottom", fontsize=9, color="firebrick",
        bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", alpha=0.8))
ax.text(0.97, 0.03, "possible DAG gap", transform=ax.transAxes,
        ha="right", va="bottom", fontsize=9, color="darkorange",
        bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", alpha=0.8))

# Quadrant counts on full data
q_ll = ((plot_df["frac_not_far_above_prec"] <= THRESHOLD_X) & (plot_df[W2] <= THRESHOLD_Y)).sum()
q_lr = ((plot_df["frac_not_far_above_prec"] > THRESHOLD_X) & (plot_df[W2] <= THRESHOLD_Y)).sum()
q_ul = ((plot_df["frac_not_far_above_prec"] <= THRESHOLD_X) & (plot_df[W2] > THRESHOLD_Y)).sum()
q_ur = ((plot_df["frac_not_far_above_prec"] > THRESHOLD_X) & (plot_df[W2] > THRESHOLD_Y)).sum()
total = len(plot_df)
print(f"Quadrant counts (n={total:,}):")
print(f"  low heuristic + low DAG  (LL): {q_ll:6,}  {100*q_ll/total:.1f}%")
print(f"  high heuristic + low DAG (LR): {q_lr:6,}  {100*q_lr/total:.1f}%  ← possible DAG gap")
print(f"  low heuristic + high DAG (UL): {q_ul:6,}  {100*q_ul/total:.1f}%")
print(f"  high heuristic + high DAG (UR): {q_ur:6,}  {100*q_ur/total:.1f}%")

handles = [plt.scatter([], [], s=20, color=COVERAGE_TIER_COLORS[t], label=t) for t in COVERAGE_TIER_ORDER]
ax.legend(handles=handles, fontsize=8, loc="center right")
ax.set_xlabel("fraction of peaks ≤ precursor + 1.5 Da")
ax.set_ylabel(f"wrecall_h2 (DAG coverage)")
ax.set_title("DAG coverage vs peak-position heuristic")
plt.tight_layout()
plt.show()

In [ ]:
# E. Molecular complexity vs wrecall
# Bin molecules by num_atoms; show median wrecall_h2 per bin + count annotation

mol_wrecall = (
    merged.dropna(subset=["num_atoms", W2])
    .groupby("mol_id")
    .agg(num_atoms=("num_atoms", "first"), med_wrecall=(W2, "median"), n_specs=(W2, "count"))
    .reset_index()
)

BINS = [0, 10, 20, 30, 40, 50, 75, 150]
LABELS = ["1-10", "11-20", "21-30", "31-40", "41-50", "51-75", "76+"]
mol_wrecall["atom_bin"] = pd.cut(mol_wrecall["num_atoms"], bins=BINS, labels=LABELS)

bin_stats = mol_wrecall.groupby("atom_bin", observed=True).agg(
    median_wrecall=("med_wrecall", "median"),
    n_mols=("mol_id", "count"),
).reset_index()

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(range(len(bin_stats)), bin_stats["median_wrecall"], color="steelblue", alpha=0.8)
for i, (_, row) in enumerate(bin_stats.iterrows()):
    ax.text(i, row["median_wrecall"] + 0.005, f"n={row['n_mols']:,}", ha="center", fontsize=8)

ax.set_xticks(range(len(bin_stats)))
ax.set_xticklabels(bin_stats["atom_bin"], fontsize=9)
ax.set_xlabel("Number of heavy atoms")
ax.set_ylabel("Median wrecall_h2 (per molecule)")
ax.set_title("Molecular complexity vs fragmentation coverage")
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## F. CMF Fix — wrecall impact for non-standard adducts

`[M+NH4]+` and `[M+Cl]-` had artificially low wrecall in the prepass because `compute_frag_peak_stats`
was adding the full adduct mass (+18 Da / +36 Da) to fragment m/z values instead of the CMF mass (+1 Da / +1 Da).

For these adducts, the charge migrates to H⁺ **before** fragmentation:
- `[M+NH4]+` → `[M+H]+` + NH₃ (−17 Da), then standard fragmentation
- `[M+Cl]-` → `[M−H]-` + HCl (−36 Da), fragments carry H⁻ not Cl⁻

Here we recompute wrecall on a sample using both the old and new (CMF-aware) mass diff to quantify the fix.

In [ ]:
import bz2
import pickle
import sys

import torch as th

sys.path.insert(0, str(REPO_ROOT / "src"))
from fragnnet.utils.formula_utils import PREC_TYPE_TO_CMF_MASS_DIFF, PREC_TYPE_TO_MASS_DIFF

DAGS_DP = MBFS_PREPASS_PATH / "dags"
SAMPLE_N = 500
TOLERANCE = 0.01
ALLOWED_H = 2  # match W2

TARGET_PREC_TYPES = ["[M+NH4]+", "[M+Cl]-", "[M+H]+", "[M-H]-"]

# Load spec metadata with peaks and prec_mz for target adducts
spec_full = pd.read_pickle(PROC_DP / "spec_df.pkl")[
    ["spec_id", "mol_id", "prec_type", "prec_mz", "peaks"]
].copy()
spec_full = spec_full[spec_full["prec_type"].isin(TARGET_PREC_TYPES)].copy()
spec_full["spec_id"] = spec_full["spec_id"].astype(int)
spec_full["mol_id"] = spec_full["mol_id"].astype(int)
print(f"Spectra with target prec_types: {len(spec_full):,}")
print(spec_full["prec_type"].value_counts())


def load_dag(mol_id: int) -> dict | None:
    path = DAGS_DP / f"{mol_id:08d}.pkl.bz2"
    if not path.exists():
        return None
    with bz2.open(path, "rb") as f:
        return pickle.load(f)


def compute_wrecall(peaks, dag: dict, prec_type: str, include_cmf: bool) -> float:
    """Compute weighted recall with standard ions or the production std ∪ CMF union."""
    if not peaks or dag is None:
        return float("nan")
    true_mzs = th.tensor([p[0] for p in peaks], dtype=th.float32)
    true_ints = th.tensor([p[1] for p in peaks], dtype=th.float32)

    formula_peak_mzs = dag["formula_peak_mzs"]
    formula_peak_probs = dag["formula_peak_probs"]
    idx_by_h_delta = dag["idx_by_h_delta"]

    allowed_idx = list(idx_by_h_delta[0])
    for h in range(1, ALLOWED_H):
        allowed_idx += list(idx_by_h_delta[2 * h - 1])
        allowed_idx += list(idx_by_h_delta[2 * h])
    allowed_idx = list(set(allowed_idx))
    indices = th.tensor(allowed_idx)
    mzs = th.index_select(formula_peak_mzs, 0, indices)
    probs = th.index_select(formula_peak_probs, 0, indices)

    prec_mask = (probs > 0.0).float()
    std_mzs = mzs + prec_mask * PREC_TYPE_TO_MASS_DIFF[prec_type]
    if include_cmf:
        cmf_diff = PREC_TYPE_TO_CMF_MASS_DIFF.get(prec_type, PREC_TYPE_TO_MASS_DIFF[prec_type])
        th_mzs = th.cat((std_mzs.reshape(-1), (mzs + prec_mask * cmf_diff).reshape(-1)))
    else:
        th_mzs = std_mzs.reshape(-1)

    matched_ints = []
    for true_mz, true_int in zip(true_mzs, true_ints):
        if th.any(th.abs(th_mzs - true_mz) < TOLERANCE):
            matched_ints.append(true_int.item())
    return float(sum(matched_ints) / true_ints.sum().item()) if len(matched_ints) > 0 else 0.0


# Sample and recompute
results = []
for prec_type in TARGET_PREC_TYPES:
    sub = spec_full[spec_full["prec_type"] == prec_type]
    sample = sub.sample(min(SAMPLE_N, len(sub)), random_state=42)
    for _, row in sample.iterrows():
        dag = load_dag(int(row["mol_id"]))
        if dag is None:
            continue
        w_old = compute_wrecall(row["peaks"], dag, prec_type, include_cmf=False)
        w_new = compute_wrecall(row["peaks"], dag, prec_type, include_cmf=True)
        results.append({"prec_type": prec_type, "wrecall_old": w_old, "wrecall_new": w_new})

cmf_df = pd.DataFrame(results)
print(f"\nRecomputed on {len(cmf_df):,} spectra")
summary = cmf_df.groupby("prec_type").agg(
    n=("wrecall_old", "count"),
    median_old=("wrecall_old", "median"),
    median_new=("wrecall_new", "median"),
).reset_index()
summary["delta"] = summary["median_new"] - summary["median_old"]
print(summary.to_string(index=False))

In [ ]:
# Plot before/after wrecall distributions per prec_type
fig, axes = plt.subplots(1, len(TARGET_PREC_TYPES), figsize=(16, 4), sharey=True)

ADDUCT_COLORS = {
    "[M+NH4]+": "tab:orange",
    "[M+Cl]-": "tab:red",
    "[M+H]+": "tab:blue",
    "[M-H]-": "tab:green",
}

for ax, prec_type in zip(axes, TARGET_PREC_TYPES):
    sub = cmf_df[cmf_df["prec_type"] == prec_type]
    color = ADDUCT_COLORS[prec_type]
    parts = ax.violinplot(
        [sub["wrecall_old"].values, sub["wrecall_new"].values],
        positions=[0, 1],
        showmedians=True,
        showextrema=False,
    )
    for i, pc in enumerate(parts["bodies"]):
        pc.set_facecolor(color)
        pc.set_alpha(0.4 if i == 0 else 0.8)
    med_old = sub["wrecall_old"].median()
    med_new = sub["wrecall_new"].median()
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["old\n(std only)", "new\n(std ∪ CMF)"], fontsize=8)
    ax.set_title(f"{prec_type}\nn={len(sub):,}", fontsize=9)
    ax.set_ylim(0, 1.05)
    ax.axhline(med_old, color=color, lw=1, ls="--", alpha=0.5)
    ax.axhline(med_new, color=color, lw=1.5, ls="-")
    delta = med_new - med_old
    sign = "+" if delta >= 0 else ""
    ax.text(0.5, 0.97, f"Δmedian={sign}{delta:.3f}", transform=ax.transAxes,
            ha="center", va="top", fontsize=8,
            color="darkgreen" if delta > 0.01 else "gray")

axes[0].set_ylabel("wrecall_h2")
fig.suptitle("wrecall before vs after CMF fix (sample of 500 per adduct)", fontsize=10)
plt.tight_layout()
plt.show()